In [1]:
!pip install -q langchain langchain-core langchain-community langchain-google-genai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 852.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.7/81.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1

In [5]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load API Key securely
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# 2. Sample Legal Knowledge Base (Multiple Contracts/Policies)
legal_documents = [
    """
    MASTER SERVICES AGREEMENT (MSA) - VENDOR A (Updated Jan 2025)
    Section 4.2: Liability Limitations. Total aggregate liability for Vendor A under this
    Agreement shall be capped at exactly $50,000 USD or the total fees paid in the preceding
    12 months, whichever is lower. Confidentiality breaches are explicitly excluded from this cap.
    """,
    """
    ENTERPRISE LICENSE AGREEMENT - VENDOR B (Updated June 2025)
    Section 8.1: Indemnification & Liability. Vendor B's total liability shall not exceed
    $250,000 USD. The confidentiality obligations under this agreement shall survive for a period
    of 7 years following termination.
    """,
    """
    DATA PROCESSING ADDENDUM (DPA) - GENERAL CLAUSES (2026 Standard)
    Section 12: Data Deletion. Upon termination or expiration of the main agreement, the processor
    shall within 30 days certify the complete destruction or return of all personal and confidential data.
    """
]

# 3. Split documents into smaller text chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = text_splitter.create_documents(legal_documents)

# 4. Create Embeddings & Store in ChromaDB Vector Database
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vector_store = Chroma.from_documents(docs, embeddings)

# 5. Create a Retriever from the Vector Database
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# 6. Initialize Gemini LLM for RAG generation
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.1)

# 7. Define RAG Prompt Template
rag_prompt = ChatPromptTemplate.from_template(
    "You are a Legal Knowledge Assistant.\n"
    "Answer the user's question accurately using *only* the provided context below. "
    "If you do not know the answer based on the context, state that you cannot find it.\n\n"
    "Context:\n{context}\n\n"
    "Question:\n{question}"
)

# Helper function to format retrieved docs
def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

# 8. Build the RAG Chain using LangChain Expression Language (LCEL)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# -------------------------------------------------------------------
# TEST THE RAG KNOWLEDGE BASE
# -------------------------------------------------------------------
query = "What is the liability cap for Vendor A?"
print(f"🔍 User Query: {query}\n")

response = rag_chain.invoke(query)
print("🤖 RAG Answer:")
print(response)

🔍 User Query: What is the liability cap for Vendor A?

🤖 RAG Answer:
Based on the provided context, the total aggregate liability for Vendor A is capped at exactly $50,000 USD or the total fees paid in the preceding (the provided text cuts off at this point).
